In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Set PLANTVILLAGE_DIR to a folder containing train/ and val/ directories.
data_dir = os.environ.get("PLANTVILLAGE_DIR")
if not data_dir:
    raise RuntimeError("Set PLANTVILLAGE_DIR before running this training notebook.")

IMG_SIZE = 224      # ResNet18 expects 224x224 images
BATCH_SIZE = 32     # how many images processed at once — 32 is a safe default for your GPU

# TRAIN transform: includes augmentation (random crop/flip/rotation/color jitter)
# Why augment: your training photos are lab-clean. The hackathon's real test is
# messy field photos. Augmentation forces the model to not rely on "perfect" conditions.
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # these numbers are fixed —
                          std=[0.229, 0.224, 0.225]),   # required to match ResNet18's pretraining
])

# VAL transform: NO augmentation — we want a clean, honest measurement of accuracy
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

# Load the folders — ImageFolder automatically uses subfolder names as class labels
train_dataset = datasets.ImageFolder(f"{data_dir}\\train", transform=train_transform)
val_dataset   = datasets.ImageFolder(f"{data_dir}\\val", transform=val_transform)

# DataLoader wraps the dataset so we can pull batches of images during training
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Classes found: {len(train_dataset.classes)}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"\nFirst 5 class names: {train_dataset.classes[:5]}")

Classes found: 38
Training images: 43444
Validation images: 10861

First 5 class names: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [2]:
import torch
import torch.nn as nn
from torchvision import models

# Use GPU since we confirmed it's available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load ResNet18 pretrained on ImageNet (1.2M general images)
# weights=IMAGENET1K_V1 downloads the pretrained weights (only needs to happen once, then cached)
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Check what the original final layer looks like BEFORE we change it
print(f"\nOriginal final layer: {model.fc}")

# Swap the final layer: in_features stays the same (512 — that's the size of
# the "features" the network extracted right before its old decision layer),
# but out_features changes from 1000 (ImageNet classes) to 38 (your classes)
num_classes = len(train_dataset.classes)  # = 38, from Cell 2
model.fc = nn.Linear(model.fc.in_features, num_classes)

print(f"New final layer: {model.fc}")

# Move the whole model onto the GPU
model = model.to(device)

print(f"\nModel ready. Will output {num_classes}-way predictions.")

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\patel/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

  1%|          | 512k/44.7M [00:00<00:09, 4.90MB/s]

  2%|▏         | 1.00M/44.7M [00:00<00:13, 3.44MB/s]

  3%|▎         | 1.50M/44.7M [00:00<00:11, 3.82MB/s]

...

 97%|█████████▋| 43.5M/44.7M [00:11<00:00, 3.91MB/s]

 99%|█████████▊| 44.0M/44.7M [00:11<00:00, 3.94MB/s]

100%|█████████▉| 44.5M/44.7M [00:12<00:00, 3.87MB/s]

100%|██████████| 44.7M/44.7M [00:12<00:00, 3.87MB/s]


Original final layer: Linear(in_features=512, out_features=1000, bias=True)
New final layer: Linear(in_features=512, out_features=38, bias=True)

Model ready. Will output 38-way predictions.


In [3]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()          # measures how wrong each guess is
optimizer = optim.Adam(model.parameters(), lr=1e-4)   # updates weights; lr = how big each nudge is

EPOCHS = 10   # how many full passes over the training data — we can adjust after seeing epoch 1's speed

print("Loss function and optimizer ready.")
print(f"Will train for {EPOCHS} epochs.")

Loss function and optimizer ready.
Will train for 10 epochs.


In [4]:
import time
from sklearn.metrics import f1_score

best_macro_f1 = -1.0
best_model_state = None

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()  # tells the model "we're learning now" (affects some internal behavior)
    running_loss = 0.0
    start_time = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)  # move batch to GPU

        optimizer.zero_grad()          # clear old gradients from last batch
        outputs = model(images)        # Step 1: model guesses
        loss = criterion(outputs, labels)  # Step 2: measure how wrong
        loss.backward()                # Step 3a: figure out which direction to nudge weights
        optimizer.step()               # Step 3b: actually nudge them

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_dataset)

    # --- VALIDATION PHASE (check progress, don't learn from this data) ---
    model.eval()  # tells the model "we're just testing now, no learning"
    all_preds, all_labels = [], []

    with torch.no_grad():  # don't bother computing gradients — we're not updating weights here
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu()  # pick the class with highest score
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    val_macro_f1 = f1_score(all_labels, all_preds, average="macro")
    elapsed = time.time() - start_time

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | "
          f"val_macro_f1={val_macro_f1:.4f} | time={elapsed:.1f}s")

    # Keep a copy of the best-performing version of the model so far
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ↳ New best! Saved checkpoint (macro-F1={best_macro_f1:.4f})")

print(f"\nTraining done. Best validation macro-F1: {best_macro_f1:.4f}")

Epoch 1/10 | train_loss=0.2728 | val_macro_f1=0.9777 | time=184.1s
  ↳ New best! Saved checkpoint (macro-F1=0.9777)


Epoch 2/10 | train_loss=0.0765 | val_macro_f1=0.9765 | time=95.1s


Epoch 3/10 | train_loss=0.0550 | val_macro_f1=0.9786 | time=94.7s
  ↳ New best! Saved checkpoint (macro-F1=0.9786)


Epoch 4/10 | train_loss=0.0441 | val_macro_f1=0.9850 | time=94.7s
  ↳ New best! Saved checkpoint (macro-F1=0.9850)


Epoch 5/10 | train_loss=0.0404 | val_macro_f1=0.9878 | time=94.2s
  ↳ New best! Saved checkpoint (macro-F1=0.9878)


Epoch 6/10 | train_loss=0.0372 | val_macro_f1=0.9867 | time=96.6s


Epoch 7/10 | train_loss=0.0303 | val_macro_f1=0.9912 | time=96.6s
  ↳ New best! Saved checkpoint (macro-F1=0.9912)


Epoch 8/10 | train_loss=0.0294 | val_macro_f1=0.9797 | time=97.8s


Epoch 9/10 | train_loss=0.0302 | val_macro_f1=0.9903 | time=97.0s


Epoch 10/10 | train_loss=0.0268 | val_macro_f1=0.9883 | time=96.9s

Training done. Best validation macro-F1: 0.9912


In [5]:
from sklearn.metrics import confusion_matrix, classification_report

# Load the BEST weights (epoch 7), not whatever's currently in `model`
# (currently model holds epoch 10's weights, since training loops don't auto-revert)
model.load_state_dict(best_model_state)
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

# Confusion matrix: rows = actual class, columns = predicted class
# Diagonal = correct predictions. Off-diagonal = mix-ups (which class got confused with which)
cm = confusion_matrix(all_labels, all_preds)

# Per-class precision/recall/F1 — this tells you WHICH classes are weak, not just an average
report = classification_report(all_labels, all_preds, target_names=train_dataset.classes, digits=3)

print("=== Per-class report ===")
print(report)
print(f"\nConfusion matrix shape: {cm.shape}")  # should be (38, 38)

=== Per-class report ===
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab      1.000     0.984     0.992       126
                                 Apple___Black_rot      1.000     1.000     1.000       125
                          Apple___Cedar_apple_rust      1.000     1.000     1.000        55
                                   Apple___healthy      1.000     0.994     0.997       329
                               Blueberry___healthy      1.000     1.000     1.000       300
          Cherry_(including_sour)___Powdery_mildew      1.000     1.000     1.000       210
                 Cherry_(including_sour)___healthy      1.000     0.988     0.994       170
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot      0.967     0.845     0.902       103
                       Corn_(maize)___Common_rust_      1.000     0.992     0.996       239
               Corn_(maize)___Northern_Leaf_Blight    

In [6]:
import pickle

bundle = {
    "architecture": "resnet18",
    "num_classes": num_classes,
    "class_names": train_dataset.classes,   # index -> label string mapping, critical for predict.py
    "state_dict": model.state_dict(),       # the actual trained weights (currently epoch 7's, since we loaded best_model_state)
    "img_size": IMG_SIZE,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
}

with open("model.pkl", "wb") as f:
    pickle.dump(bundle, f)

print("Saved model.pkl")
print(f"File size check:")

import os
size_mb = os.path.getsize("model.pkl") / (1024 * 1024)
print(f"model.pkl is {size_mb:.1f} MB")

Saved model.pkl
File size check:
model.pkl is 42.8 MB


In [7]:
import pickle
import torch
import torch.nn as nn
from torchvision import models

with open("model.pkl", "rb") as f:
    loaded_bundle = pickle.load(f)

print(f"Loaded bundle — architecture: {loaded_bundle['architecture']}, classes: {loaded_bundle['num_classes']}")

# Rebuild the model architecture from scratch (fresh object, no shared state with the original `model`)
fresh_model = models.resnet18(weights=None)  # weights=None: skip re-downloading ImageNet weights, we're loading our own
fresh_model.fc = nn.Linear(fresh_model.fc.in_features, loaded_bundle["num_classes"])
fresh_model.load_state_dict(loaded_bundle["state_dict"])
fresh_model = fresh_model.to(device)
fresh_model.eval()

# Re-run validation with the freshly loaded model
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = fresh_model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

fresh_macro_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"\nFresh-loaded model macro-F1: {fresh_macro_f1:.4f}")
print(f"Original in-memory macro-F1: {best_macro_f1:.4f}")
print(f"Match: {'YES' if abs(fresh_macro_f1 - best_macro_f1) < 0.0001 else 'MISMATCH — something went wrong in saving'}")

Loaded bundle — architecture: resnet18, classes: 38



Fresh-loaded model macro-F1: 0.9912
Original in-memory macro-F1: 0.9912
Match: YES


In [8]:
import datetime

print("=" * 70)
print("AGRISMART AI — MODEL SUMMARY REPORT")
print("=" * 70)

# --- Task & dataset ---
print(f"\n[TASK]")
print(f"  Crop-disease image classification, {num_classes} classes")

print(f"\n[DATASET]")
print(f"  Source: PlantVillage (color, lab-condition images)")
print(f"  Training images: {len(train_dataset)}")
print(f"  Validation images: {len(val_dataset)}")
print(f"  Total: {len(train_dataset) + len(val_dataset)}")
print(f"  Split: pre-split train/val (~80/20), provided as-is")

# --- Model / approach ---
print(f"\n[MODEL / APPROACH]")
print(f"  Architecture: ResNet18 (transfer learning, ImageNet-pretrained)")
print(f"  Input size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Optimizer: Adam, lr=1e-4")
print(f"  Loss: CrossEntropyLoss")
print(f"  Epochs trained: {EPOCHS}")
print(f"  Best epoch: 7 (by val macro-F1)")
print(f"  Augmentation: random crop, flip, rotation, color jitter")

# --- Headline metrics ---
overall_accuracy = (torch.tensor(all_preds) == torch.tensor(all_labels)).float().mean().item()
print(f"\n[HEADLINE METRICS] (validation set)")
print(f"  Macro-F1 (primary metric): {best_macro_f1:.4f}")
print(f"  Overall accuracy: {overall_accuracy:.4f}")

# --- Weakest classes (worth naming in Limitations) ---
from sklearn.metrics import f1_score as f1_per_class
per_class_f1 = f1_per_class(all_labels, all_preds, average=None)
class_f1_pairs = list(zip(train_dataset.classes, per_class_f1))
class_f1_pairs.sort(key=lambda x: x[1])  # weakest first

print(f"\n[WEAKEST 5 CLASSES] (lowest F1 — flag these as honest limitations)")
for name, f1 in class_f1_pairs[:5]:
    print(f"  {name}: F1={f1:.3f}")

print(f"\n[STRONGEST 5 CLASSES]")
for name, f1 in class_f1_pairs[-5:]:
    print(f"  {name}: F1={f1:.3f}")

# --- Files produced ---
print(f"\n[ARTIFACTS]")
print(f"  model.pkl saved: {os.path.exists('model.pkl')}")
if os.path.exists("model.pkl"):
    print(f"  model.pkl size: {os.path.getsize('model.pkl')/(1024*1024):.1f} MB")

print(f"\n[GENERATED]")
print(f"  {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

# --- Also save this to a text file for the /report folder ---
summary_text = f"""AgriSmart AI — Model Summary Report
Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

TASK: Crop-disease image classification, {num_classes} classes

DATASET
  Source: PlantVillage (color, lab-condition images)
  Training images: {len(train_dataset)}
  Validation images: {len(val_dataset)}
  Split: pre-split train/val (~80/20)

MODEL
  Architecture: ResNet18 (transfer learning, ImageNet-pretrained)
  Input size: {IMG_SIZE}x{IMG_SIZE} | Batch size: {BATCH_SIZE}
  Optimizer: Adam (lr=1e-4) | Loss: CrossEntropyLoss
  Epochs trained: {EPOCHS} | Best epoch: 7

HEADLINE METRICS (validation set)
  Macro-F1: {best_macro_f1:.4f}
  Accuracy: {overall_accuracy:.4f}

WEAKEST CLASSES
{chr(10).join(f"  {name}: F1={f1:.3f}" for name, f1 in class_f1_pairs[:5])}

FULL PER-CLASS REPORT
{report}
"""

with open("model_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved full report to model_summary.txt")

AGRISMART AI — MODEL SUMMARY REPORT

[TASK]
  Crop-disease image classification, 38 classes

[DATASET]
  Source: PlantVillage (color, lab-condition images)
  Training images: 43444
  Validation images: 10861
  Total: 54305
  Split: pre-split train/val (~80/20), provided as-is

[MODEL / APPROACH]
  Architecture: ResNet18 (transfer learning, ImageNet-pretrained)
  Input size: 224x224
  Batch size: 32
  Optimizer: Adam, lr=1e-4
  Loss: CrossEntropyLoss
  Epochs trained: 10
  Best epoch: 7 (by val macro-F1)
  Augmentation: random crop, flip, rotation, color jitter

[HEADLINE METRICS] (validation set)
  Macro-F1 (primary metric): 0.9912
  Overall accuracy: 0.9943

[WEAKEST 5 CLASSES] (lowest F1 — flag these as honest limitations)
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: F1=0.902
  Corn_(maize)___Northern_Leaf_Blight: F1=0.954
  Potato___healthy: F1=0.969
  Peach___healthy: F1=0.973
  Tomato___Early_blight: F1=0.980

[STRONGEST 5 CLASSES]
  Potato___Early_blight: F1=1.000
  Rasp